# RAG jurídico em português: LGPD, Marco Civil e CDC

Rode as células em ordem. Use **GPU** (Ambiente de execução → Alterar tipo → T4) para a etapa de geração.

In [ ]:
# 1) Envie o legal-rag-pt.zip quando o seletor de arquivos aparecer
from google.colab import files
up = files.upload()
zip_name = next(iter(up))
!unzip -q -o "{zip_name}"
%cd legal-rag-pt

In [ ]:
!pip install -q -r requirements.txt

## Ingestão
Baixa as 3 leis do Planalto e divide por artigo.

In [ ]:
!python src/ingest.py
import json
chunks = [json.loads(l) for l in open('data/chunks.jsonl', encoding='utf-8')]
print(len(chunks), 'chunks')
for c in chunks[:2]:
    print('\n', c['chunk_id'], '\n', c['text'][:300])

**Confira a ingestão:** o número de artigos por lei deve ser próximo do real (LGPD ≈ 65, Marco Civil ≈ 32, CDC ≈ 119 artigos em vigor). Se vier muito diferente, o parser precisa de ajuste.

In [ ]:
!python tests/test_core.py

## Avaliação da recuperação

In [ ]:
!python src/eval_retrieval.py
import pandas as pd
pd.read_csv('results/retrieval_results.csv')

In [ ]:
# Perguntas em que cada recuperador errou (artigo correto fora do top 5)
pd.read_csv('results/retrieval_failures.csv').head(30)

## Geração e avaliação das respostas
Baixa o modelo `Qwen/Qwen2.5-3B-Instruct` (precisa de GPU).

In [ ]:
!python src/eval_generation.py --k 5
pd.read_csv('results/generation_results.csv')[['id','question','gold','cited','citation_hit','citation_valid','grounded_sim','abstained']]

In [ ]:
# Baixa os resultados (para o README e para a revisão manual)
!zip -q -r resultados_rag.zip results data/chunks.jsonl
files.download('resultados_rag.zip')